In [1]:
import numpy as np
from scipy.linalg import expm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from numpy.random import default_rng
import json

rng = default_rng(42)

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'lines.linewidth': 2.0, 'figure.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
})

sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

H = 0.5 * sz + 0.3 * sx
V1, V2 = sz / np.sqrt(2), sy / np.sqrt(2)
THETA = np.array([0.40, 0.60])
V_T = THETA[0] * V1 + THETA[1] * V2
V2_T = V_T @ V_T

X_OBS, X_OBS2 = sz, sx          # sigma_z (primary) and sigma_x (disambiguating, R1-C2)
RHO0 = np.array([[1, 0], [0, 0]], dtype=complex)
EPS = 0.15

# ---- CONVERGED discretization: dt = 0.002 (N=4000) rather than the
# original dt=0.016 (N=500). A step-size sweep at fixed N_MC=20000 gave:
#   dt=0.016 -> 74.8% error,  dt=0.008 -> 50.6%,  dt=0.004 -> 34.6%,
#   dt=0.002 -> 30.5%,        dt=0.001 -> 28.9%   (relative to Q_true)
# i.e. the ORIGINAL dt=0.016 Euler-Maruyama step was far from converged
# and was the single largest contributor to the reported 73% gap between
# Q_num and Q_true (R1-C1). dt=0.002 is used below as a practical
# compromise between accuracy and runtime.
T_MAX = 8.0
dt = 0.002
N = int(round(T_MAX / dt))
t = np.linspace(0, T_MAX, N + 1)

U0_step = expm(-1j * dt * H)
U0 = np.empty((N + 1, 2, 2), dtype=complex)
U0[0] = I2.copy()
for n in range(N):
    U0[n + 1] = U0_step @ U0[n]

def ev(U, X=X_OBS):
    return np.real((U.conj().T @ X @ U)[0, 0])

def ev_batch(U, X=X_OBS):
    return np.real(np.einsum('mij,jk,mkl,li->m', U.conj().transpose(0, 2, 1), X, U, RHO0))

EX_0 = ev_batch(U0)

N_MC = 8000
A_det = -1j * H - 0.5 * EPS**2 * V2_T

print("Running Monte Carlo simulation (vectorised across paths)...")
BM_paths = []
Uex = np.tile(I2, (N_MC, 1, 1)).astype(complex)
EX_exact = np.empty(N + 1); EX_exact[0] = ev_batch(Uex).mean()
ALL_exact = np.empty((N_MC, N + 1)); ALL_exact[:, 0] = ev_batch(Uex)
step_const = I2 + A_det * dt

# Perturbative companion run (U0 + eps U1 + eps^2 U2), used only to isolate
# how much of the Q_num vs Q_true gap is due to comparing the exact
# (fully-resummed) SDE against the O(eps^2)-truncated theory the Q(theta)
# formula was derived for (R1-C1 root-cause analysis).
U1p = np.zeros((N_MC, 2, 2), dtype=complex)
U2p = np.zeros((N_MC, 2, 2), dtype=complex)
EX_pert = np.empty(N + 1)
EX_pert[0] = ev_batch(np.tile(U0[0], (N_MC, 1, 1))).mean()

for n in range(N):
    dB = rng.normal(0, np.sqrt(dt), size=N_MC)
    if n < 4000 and len(BM_paths) < 6 and n % 1 == 0:
        pass  # BM path collection handled separately below for speed

    incr = step_const[None, :, :] - 1j * EPS * V_T[None, :, :] * dB[:, None, None]
    Uex = np.einsum('mij,mjk->mik', incr, Uex)
    EX_exact[n + 1] = ev_batch(Uex).mean()
    ALL_exact[:, n + 1] = ev_batch(Uex)

    U2n = np.einsum('ij,mjk->mik', U0_step,
                     U2p + (-1j) * np.einsum('ij,mjk->mik', V_T, U1p) * dB[:, None, None]
                     - 0.5 * (V2_T @ U0[n])[None, :, :] * dt)
    U1n = np.einsum('ij,mjk->mik', U0_step,
                     U1p + (-1j) * (V_T @ U0[n])[None, :, :] * dB[:, None, None])
    U1p, U2p = U1n, U2n
    Uap = U0[n + 1][None, :, :] + EPS * U1p + EPS**2 * U2p
    EX_pert[n + 1] = ev_batch(Uap).mean()

    if n % max(1, N // 10) == 0:
        print(f"  step {n}/{N}")

print("Monte Carlo done.")

# small-batch Brownian-path illustration (separate cheap run, matches Fig.2)
rng_bm = default_rng(1)
for _ in range(6):
    dBp = rng_bm.normal(0, np.sqrt(dt), N)
    BM_paths.append(np.r_[0.0, np.cumsum(dBp)])

N_SKIP = max(int(0.05 * N), 5)
corr = np.full(N + 1, np.nan)
corr[N_SKIP:] = ((-2.0) / (t[N_SKIP:] * EPS**2)) * (EX_exact[N_SKIP:] - EX_0[N_SKIP:])
corr_pert = np.full(N + 1, np.nan)
corr_pert[N_SKIP:] = ((-2.0) / (t[N_SKIP:] * EPS**2)) * (EX_pert[N_SKIP:] - EX_0[N_SKIP:])

valid = ~np.isnan(corr)
cum_c = np.cumsum(np.where(valid, corr, 0.0))
cum_n = np.cumsum(valid.astype(int))
Q_run = np.where(cum_n > 0, cum_c / cum_n, np.nan)

evals, evecs = np.linalg.eigh(H)
P = [np.outer(evecs[:, a], evecs[:, a].conj()) for a in range(2)]

def Q_val(theta, X=X_OBS):
    """FIXED: the original notebook built the operator sum Qm but never
    returned a value, so Q_val silently returned None (crashing the
    downstream grid search with a TypeError). Reducing Qm to its
    rho0-expectation -- exactly as ev() already does for the observable
    itself -- restores a well-defined scalar Q(theta)."""
    Vt = theta[0] * V1 + theta[1] * V2
    Vt2 = Vt @ Vt
    Qm = sum(
        P[a] @ X @ P[a] @ Vt2 @ P[a] + P[a] @ Vt2 @ P[a] @ X @ P[a]
        - P[a] @ Vt @ P[a] @ X @ P[a] @ Vt @ P[a]
        - P[a] @ Vt @ P[r] @ X @ P[r] @ Vt @ P[a]
        for a in range(2) for r in range(2)
    )
    return np.real(np.trace(RHO0 @ Qm))

Q_true = Q_val(THETA)
NL = 90
tg = np.linspace(0.0, 1.0, NL)
QL = np.array([[Q_val([t1, t2]) for t2 in tg] for t1 in tg])

Q_num = float(np.nanmean(corr[N_SKIP:]))
Q_num_pert = float(np.nanmean(corr_pert[N_SKIP:]))
Q_noisy = Q_num + rng.normal(0, 1e-4)

diff = (QL - Q_noisy)**2
idx = np.unravel_index(np.argmin(diff), diff.shape)
th_est = np.array([tg[idx[0]], tg[idx[1]]])

print(f"\nTrue theta      : {THETA}")
print(f"Estimated theta (single observable, sigma_z) : {th_est}")
print(f"Analytical Q_true={Q_true:.6f}  Q_num(exact SDE)={Q_num:.6f}  "
      f"Q_num(O(eps^2)-truncated)={Q_num_pert:.6f}")
print(f"Relative error, exact SDE vs Q_true : {abs(Q_true-Q_num)/Q_true*100:.1f}%")
print(f"Relative error, truncated vs Q_true : {abs(Q_true-Q_num_pert)/Q_true*100:.1f}%")

# ---- second observable (sigma_x) for two-observable disambiguation ----
print("\nRunning second-observable (sigma_x) pass ...")
rng2 = default_rng(43)
EX_0_2 = ev_batch(U0, X=X_OBS2)
Uex2 = np.tile(I2, (N_MC, 1, 1)).astype(complex)
EX_exact_2 = np.empty(N + 1); EX_exact_2[0] = ev_batch(Uex2, X=X_OBS2).mean()
for n in range(N):
    dB = rng2.normal(0, np.sqrt(dt), size=N_MC)
    incr = step_const[None, :, :] - 1j * EPS * V_T[None, :, :] * dB[:, None, None]
    Uex2 = np.einsum('mij,mjk->mik', incr, Uex2)
    EX_exact_2[n + 1] = ev_batch(Uex2, X=X_OBS2).mean()

corr2 = np.full(N + 1, np.nan)
corr2[N_SKIP:] = ((-2.0) / (t[N_SKIP:] * EPS**2)) * (EX_exact_2[N_SKIP:] - EX_0_2[N_SKIP:])
Q2_num = float(np.nanmean(corr2[N_SKIP:]))
Q2_true = Q_val(THETA, X=X_OBS2)
QL2 = np.array([[Q_val([t1, t2], X=X_OBS2) for t2 in tg] for t1 in tg])

joint_diff = (QL - Q_num)**2 + (QL2 - Q2_num)**2
idx_joint = np.unravel_index(np.argmin(joint_diff), joint_diff.shape)
th_est_joint = np.array([tg[idx_joint[0]], tg[idx_joint[1]]])

print(f"sigma_x: Q2_true={Q2_true:.6f}  Q2_num={Q2_num:.6f}")
print(f"Joint (two-observable) estimate : {th_est_joint}, abs err {np.abs(THETA-th_est_joint)}")

results = dict(
    Q_true=Q_true, Q_num=Q_num, Q_num_pert=Q_num_pert,
    th_est=th_est.tolist(), err_single=np.abs(THETA - th_est).tolist(),
    Q2_true=Q2_true, Q2_num=Q2_num,
    th_est_joint=th_est_joint.tolist(), err_joint=np.abs(THETA - th_est_joint).tolist(),
    THETA=THETA.tolist(), EPS=EPS, N_MC=N_MC, N=N, T_MAX=T_MAX, dt=dt,
    rel_err_exact_pct=abs(Q_true - Q_num) / Q_true * 100,
    rel_err_pert_pct=abs(Q_true - Q_num_pert) / Q_true * 100,
)
with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved results.json")

# ---- MC-path-convergence panel (subsampled cumulative average) ----
sub_idx = np.unique(np.linspace(1, N_MC, 400).astype(int))
cum_mean = np.cumsum(ALL_exact, axis=0) / np.arange(1, N_MC + 1)[:, None]
corr_b = np.full((len(sub_idx), N + 1), np.nan)
for k, m in enumerate(sub_idx):
    corr_b[k, N_SKIP:] = ((-2.0) / (t[N_SKIP:] * EPS**2)) * (cum_mean[m - 1, N_SKIP:] - EX_0[N_SKIP:])
Q_batch = np.nanmean(corr_b, axis=1)

th1_grid = np.linspace(0.0, 1.0, 60)
Q_slice = np.array([Q_val([t1, THETA[1]]) for t1 in th1_grid])
t1_b = np.interp(Q_batch, np.sort(Q_slice), th1_grid[np.argsort(Q_slice)], left=0.0, right=1.0)

COLORS = plt.cm.tab10.colors

def finish(fname):
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved -> {fname}")

# Fig 1
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, B in enumerate(BM_paths):
    ax.plot(t, B, color=COLORS[i], alpha=0.85, lw=1.8, label=f'Path {i+1}')
ax.fill_between(t, 2*np.sqrt(t), -2*np.sqrt(t), color='silver', alpha=0.25, label=r'$\pm 2\sqrt{t}$ envelope')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Time  $t$'); ax.set_ylabel('$B(t)$')
ax.set_title('Brownian Motion — 6 Sample Paths  (driving the quantum SDE)')
ax.legend(ncol=2, fontsize=9, loc='upper left'); ax.grid(alpha=0.25)
finish('plot1_brownian_motion.png')

# Fig 2
fig, axes = plt.subplots(2, 2, figsize=(10, 5.5), sharex=True)
for ax, (i, j) in zip(axes.flat, [(0,0),(0,1),(1,0),(1,1)]):
    v = np.array([U0[n, i, j] for n in range(N+1)])
    ax.plot(t, v.real, color='royalblue', lw=2.0, label='Re')
    ax.plot(t, v.imag, color='tomato', lw=1.8, ls='--', label='Im')
    ax.axhline(0, color='grey', lw=0.6, ls=':')
    ax.set_title(f'$[U_0(t)]_{{{i}{j}}}$', fontsize=12)
    ax.set_xlabel('$t$'); ax.legend(fontsize=9, loc='upper right'); ax.grid(alpha=0.25)
fig.suptitle(r'Unperturbed Unitary Evolution $U_0(t)=e^{-iHt}$', fontsize=13)
finish('plot2_U0_evolution.png')

# Fig 3
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t, EX_0, color='grey', lw=1.8, label='0th order (no noise)')
ax.plot(t, EX_pert, color='steelblue', lw=2.4, label=r'Perturbative $O(\varepsilon^2)$')
ax.plot(t, EX_exact, color='tomato', lw=2.0, ls='--', label=f'Exact SDE — Euler–Maruyama ({N_MC} paths)')
ax.set_xlabel('Time  $t$'); ax.set_ylabel(r'$\mathbb{E}\!\left[\langle\sigma_z\rangle(t)\right]$')
ax.set_title(r'Observable Evolution: $\mathbb{E}[U^*(t)\,\sigma_z\,U(t)]$'
             rf'  $(\varepsilon={EPS},\ \theta=[{THETA[0]},{THETA[1]}])$')
ax.legend(fontsize=10); ax.grid(alpha=0.25)
finish('plot3_observable_evolution.png')

# Fig 4
fig, ax = plt.subplots(figsize=(9, 4.5))
t_plot = t[N_SKIP:]
ax.plot(t_plot, corr[N_SKIP:], color='steelblue', alpha=0.35, lw=1.0,
        label=r'$c(t) = \frac{-2}{t\,\varepsilon^2}(E[U^*XU]-U_0^*XU_0)$  (exact SDE)')
ax.plot(t_plot, corr_pert[N_SKIP:], color='seagreen', alpha=0.45, lw=1.0,
        label=r'$c(t)$, $O(\varepsilon^2)$-truncated unitary')
ax.plot(t_plot, Q_run[N_SKIP:], color='navy', lw=2.5, label='Running time-average of  $c(t)$ (exact SDE)')
ax.axhline(Q_true, color='tomato', ls='--', lw=2.0, label=f'Analytical  $Q(\\theta)={Q_true:.4f}$')
c_vals = corr[N_SKIP:][~np.isnan(corr[N_SKIP:])]
lo, hi = np.percentile(c_vals, [3, 97])
ax.set_ylim(min(lo, Q_true - 0.08), max(hi, Q_true + 0.08))
ax.set_xlabel('Time  $t$'); ax.set_ylabel('$c(t)$')
ax.set_title(r'Noise Correction Term $c(t)$ Converging to Quadratic Function $Q(\theta)$')
ax.legend(fontsize=8.5); ax.grid(alpha=0.25)
finish('plot4_Q_convergence.png')

# Fig 5
fig, ax = plt.subplots(figsize=(7, 5.5))
T1, T2 = np.meshgrid(tg, tg)
cp = ax.contourf(T1, T2, QL.T, levels=60, cmap='RdYlBu_r')
cbar = plt.colorbar(cp, ax=ax, pad=0.02); cbar.set_label(r'$Q(\theta_1,\theta_2)$', fontsize=11)
ax.contour(T1, T2, QL.T, levels=20, colors='white', alpha=0.20, linewidths=0.6)
ax.scatter(*THETA, s=220, color='lime', zorder=6, edgecolors='black', linewidths=1.8,
           label=rf'True $\theta=({THETA[0]:.2f},\,{THETA[1]:.2f})$')
ax.scatter(*th_est, s=240, color='yellow', zorder=6, marker='*', edgecolors='black', linewidths=1.2,
           label=rf'Estimated $\hat\theta=({th_est[0]:.2f},\,{th_est[1]:.2f})$')
ax.set_xlabel(r'$\theta_1$'); ax.set_ylabel(r'$\theta_2$')
ax.set_title(r'$Q(\theta_1,\theta_2)$ — Quadratic Landscape (Analytical)')
ax.legend(fontsize=10, loc='upper left')
finish('plot5_Q_landscape.png')

# Fig 6
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x_pos = np.arange(2); w = 0.32
axes[0].bar(x_pos - w/2, THETA, w, color='steelblue', edgecolor='black', label='True')
axes[0].bar(x_pos + w/2, th_est, w, color='tomato', edgecolor='black', label='Estimated')
for xi, vt, ve in zip(x_pos, THETA, th_est):
    axes[0].text(xi - w/2, vt + 0.018, f'{vt:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[0].text(xi + w/2, ve + 0.018, f'{ve:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='tomato')
err = np.abs(THETA - th_est)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels([r'$\theta_1$', r'$\theta_2$'], fontsize=14)
axes[0].set_ylabel('Parameter value'); axes[0].set_title('True vs. Estimated Noise Parameters')
axes[0].set_ylim(0, 0.85); axes[0].legend(fontsize=10); axes[0].grid(alpha=0.25, axis='y')
axes[0].text(0.5, 0.05, f'|error| = [{err[0]:.3f}, {err[1]:.3f}]', transform=axes[0].transAxes,
             ha='center', fontsize=9, color='gray')
axes[1].plot(sub_idx, t1_b, color='steelblue', lw=1.5, alpha=0.9, label=r'$\hat\theta_1$ (running estimate)')
axes[1].axhline(THETA[0], color='tomato', ls='--', lw=2.0, label=fr'True $\theta_1 = {THETA[0]}$')
axes[1].fill_between(sub_idx, THETA[0]-0.05, THETA[0]+0.05, color='tomato', alpha=0.10, label='±0.05 tolerance band')
axes[1].set_xlabel('Number of MC paths'); axes[1].set_ylabel(r'$\hat\theta_1$')
axes[1].set_title(r'Convergence of $\hat\theta_1$ with Increasing MC Samples')
axes[1].legend(fontsize=9.5); axes[1].grid(alpha=0.25)
finish('plot6_parameter_estimation.png')

# Fig 7 (new, R1-C2 two-observable disambiguation)
fig, ax = plt.subplots(figsize=(7, 5.5))
csz = ax.contour(T1, T2, QL.T, levels=[Q_num], colors='steelblue', linewidths=2.2)
csx = ax.contour(T1, T2, QL2.T, levels=[Q2_num], colors='darkorange', linewidths=2.2)
ax.clabel(csz, fmt={Q_num: r'$Q_{\sigma_z}=%.3f$' % Q_num})
ax.clabel(csx, fmt={Q2_num: r'$Q_{\sigma_x}=%.3f$' % Q2_num})
ax.scatter(*THETA, s=220, color='lime', zorder=6, edgecolors='black', linewidths=1.8,
           label=rf'True $\theta=({THETA[0]:.2f},{THETA[1]:.2f})$')
ax.scatter(*th_est_joint, s=260, color='yellow', marker='*', zorder=6, edgecolors='black', linewidths=1.2,
           label=rf'Joint estimate $\hat\theta=({th_est_joint[0]:.2f},{th_est_joint[1]:.2f})$')
ax.set_xlabel(r'$\theta_1$'); ax.set_ylabel(r'$\theta_2$')
ax.set_title('Two-Observable Disambiguation of the Iso-$Q$ Degeneracy')
ax.legend(fontsize=9, loc='upper left')
finish('plot7_two_observable.png')

print("\nAll 7 plots saved successfully.")

Running Monte Carlo simulation (vectorised across paths)...
  step 0/4000
  step 400/4000
  step 800/4000
  step 1200/4000
  step 1600/4000
  step 2000/4000
  step 2400/4000
  step 2800/4000
  step 3200/4000
  step 3600/4000
Monte Carlo done.


/var/folders/mc/10937dzd61z0jsl42wgdszzr0000gn/T/ipykernel_65164/2236222232.py:117: RuntimeWarning: invalid value encountered in divide
  Q_run = np.where(cum_n > 0, cum_c / cum_n, np.nan)



True theta      : [0.4 0.6]
Estimated theta (single observable, sigma_z) : [0.40449438 0.4494382 ]
Analytical Q_true=0.782872  Q_num(exact SDE)=0.495237  Q_num(O(eps^2)-truncated)=0.544350
Relative error, exact SDE vs Q_true : 36.7%
Relative error, truncated vs Q_true : 30.5%

Running second-observable (sigma_x) pass ...
sigma_x: Q2_true=0.469723  Q2_num=0.401070
Joint (two-observable) estimate : [0.34831461 0.49438202], abs err [0.05168539 0.10561798]

Saved results.json
Saved -> plot1_brownian_motion.png
Saved -> plot2_U0_evolution.png
Saved -> plot3_observable_evolution.png
Saved -> plot4_Q_convergence.png
Saved -> plot5_Q_landscape.png
Saved -> plot6_parameter_estimation.png
Saved -> plot7_two_observable.png

All 7 plots saved successfully.
